# ข้อ 1: เซ็นเซอร์ที่ "ดีกว่า" คุ้มค่าจริงไหม
ฝ่ายจัดซื้อเสนอเซ็นเซอร์รุ่นใหม่ที่ false alarm ลดจาก 10% เหลือ 3% และ detection เพิ่มจาก 80% เป็น 90% ราคาแพงกว่าเดิม 3 เท่า ทีมของคุณต้องตอบว่าคุ้มไหม

1.สร้าง pump_v2 ด้วย sensor model ใหม่ แล้วรัน filtering บน alerts ชุดเดิม ระบบสั่งหยุดซ่อม (ตามเกณฑ์ MEU) เร็วขึ้นกี่กะ  
2.รันทั้งสองเซ็นเซอร์บนลำดับ false alarm [F, F, T, F, F, F] ซึ่งเป็นกรณีที่เซ็นเซอร์เตือนผิดครั้งเดียวแล้วเงียบ นับว่าแต่ละเซ็นเซอร์ทำให้ระบบ สั่งซ่อมเก้อกี่กะ  
3.สรุปว่าควรซื้อไหม พร้อมเหตุผลจากตัวเลขทั้งสองข้อ  
คำใบ้: maint_sensor แถวแรกคือ P(alert=True | [Healthy, Degraded]) แถวสองคือส่วนเติมเต็มให้รวมเป็น 1 ผลลัพธ์อาจไม่เป็นไปตามที่คาด ให้เชื่อตัวเลข

In [ ]:
# ข้อ 1: เติมโค้ดตรง TODO
# TODO: สร้าง sensor model ใหม่ตามสเปกเซ็นเซอร์รุ่นใหม่
sensor_v2 = \
[
    [0.03, 0.90], 
    [0.97, 0.10]
]

pump_v2 = HiddenMarkovModel(maint_transition, sensor_v2, maint_prior)


def run_filter(hmm, seq):
    b = maint_prior
    result = []
    
    for e in seq:
        b = forward(hmm, b, e)
        result.append(float(b[1]))

    return result


def repair_shifts(beliefs):
    return [t for t, p in enumerate(beliefs, 1) if meu(p)[1] == 'repair']


filtered_v2 = run_filter(pump_v2, alerts)
assert filtered_v2, 'ยังไม่ได้เติม TODO: run_filter ต้องคืนลิสต์ P(Degraded)'

print('1) การตรวจจับของจริง')
print('   เซ็นเซอร์เดิม  สั่งซ่อมที่กะ', repair_shifts(filtered))
print('   รุ่นใหม่       สั่งซ่อมที่กะ', repair_shifts(filtered_v2))
print('   ความมั่นใจที่กะ 4: เดิม %.3f -> ใหม่ %.3f' % (filtered[3], filtered_v2[3]))

FALSE_ALARM = [F, F, T, F, F, F]
print('\n2) กรณีเซ็นเซอร์เตือนผิดครั้งเดียว')
for name, hmm in [('เดิม  ', pump), ('ใหม่  ', pump_v2)]:
    beliefs = run_filter(hmm, FALSE_ALARM)
    print('   %s P(Degraded) = %s -> สั่งซ่อมเก้อที่กะ %s'
          % (name, ['%.3f' % p for p in beliefs], repair_shifts(beliefs)))

# 3) ตอบ: ควรซื้อไหม => ข้อมูลยังไม่พอตัดสินใจ เพราะ ต้องดูต้นทุนรวม ต้องทดสอบมากขึ้นแล้วเปรียบเทียบ

# ข้อ 2: เลือกค่า lag ให้ dashboard
คำนวณ ค่าคลาดเคลื่อนสัมบูรณ์เฉลี่ย ระหว่างรายงานที่ lag = d กับค่า smoothing เต็มรูป (ถือว่า smoothing เต็มรูปคือ "ความจริงที่ดีที่สุดที่เรารู้ได้") สำหรับ d = 0, 1, 2, 3

แล้วตอบว่า lag เท่าไรที่คุ้มที่สุด ถ้าทุก 1 กะที่หน่วงเพิ่ม มีต้นทุนเทียบเท่าค่าคลาดเคลื่อน 0.05

คำใบ้: ใช้ fixed_lag(pump, alerts[:t], d) เทียบกับ smoothed[t - d - 1] เฉพาะ t ที่ค่าไม่เป็น None

In [ ]:
# ข้อ 2: เติมโค้ดตรง TODO
LAG_PENALTY = 0.05

for d in range(4):
    errs = []

    for t in range(1, len(alerts) + 1):
        value = fixed_lag(pump, alerts[:t], d)
        
        if value is not None: errs.append(abs(value - smoothed[t - d - 1]))

    mean_err = sum(errs) / len(errs) if errs else float('nan')

    print('d=%d  mean|error| = %.4f  ต้นทุนรวม = %.4f'
          % (d, mean_err, mean_err + d * LAG_PENALTY))

# ตอบ: lag ที่คุ้มที่สุดคือ d = 2

### ข้อ 3: ตรวจจับ particle depletion

เพิ่มการคำนวณ **effective sample size** $ESS = 1 / \sum_i w_i^2$ ลงใน particle filter
ค่านี้บอกว่า "จริง ๆ แล้วมีอนุภาคกี่ตัวที่มีส่วนร่วมในการประมาณ"
ถ้า ESS ร่วงต่ำกว่า N/2 แปลว่าอนุภาคส่วนใหญ่แทบไม่มีส่วนร่วม ค่าประมาณเริ่มเชื่อไม่ได้

จากนั้นจำลองสถานการณ์ **น้ำมันหล่อลื่นรั่ว** โดยให้ ground truth สึกหรอเร็วขึ้น 3 เท่า
(`WEAR_STEP_MEAN * 3` ใน `simulate_pump` เท่านั้น ห้ามแก้ใน filter)
แล้วเทียบ ESS กับกรณีปกติ (`simulate_pump()`) ว่าร่วงตอนไหนและร่วงแค่ไหน

*คำใบ้:* คำนวณ ESS จาก `weights` หลัง normalize แล้ว ก่อนขั้น resample

In [ ]:
# ข้อ 3: เติมโค้ดตรง TODO
def simulate_pump_leak(n_shifts=9, seed=7, drift=2.0):
    random.seed(seed)
    w, truth, obs = 0.05, [], []
    for _ in range(n_shifts):
        w = min(1.0, w + max(0.0, random.gauss(WEAR_STEP_MEAN * drift, WEAR_STEP_SD)))
        truth.append(w)
        obs.append(TEMP_BASE + TEMP_GAIN * w + random.gauss(0, TEMP_SD))
    return truth, obs


def wear_pf_with_ess(obs, N=3000, seed=1):
    random.seed(seed)
    particles = [random.uniform(0.0, 0.12) for _ in range(N)]
    mean, ess = [], []
    for z in obs:
        particles = [min(1.0, w + max(0.0, random.gauss(WEAR_STEP_MEAN, WEAR_STEP_SD)))
                     for w in particles]
        weights = [gaussian(TEMP_BASE + TEMP_GAIN * w, TEMP_SD, z) for w in particles]
        total = sum(weights)
        weights = [w / total for w in weights] if total > 0 else [1.0 / N] * N
        ess.append(1.0 / sum(w * w for w in weights))
        particles = weighted_sample_with_replacement(N, particles, weights)
        mean.append(sum(particles) / N)
    return mean, ess


truth_leak, obs_leak = simulate_pump_leak()
est_leak, ess_leak = wear_pf_with_ess(obs_leak)

print('%3s %9s %9s %10s %s' % ('กะ', 'จริง', 'ประมาณ', 'ESS', 'สถานะ'))
for t in range(len(truth_leak)):
    warn = 'LOW ESS' if ess_leak[t] == ess_leak[t] and ess_leak[t] < 1500 else ''
    print('%3d %9.3f %9.3f %10.1f %s' % (t + 1, truth_leak[t], est_leak[t], ess_leak[t], warn))

# ข้อ 4: AGV หลงทาง
รัน MCL ด้วย การสแกนที่ผิดพลาด คือ z = (9, 9, 9, 9) ซึ่งเป็นค่าที่แทบเป็นไปไม่ได้บนแผนที่นี้ (เช่น LiDAR สกปรกหรือมีคนเดินบัง)

1.เกิดอะไรขึ้นกับการกระจายของอนุภาค  
2.ระบบจริงควรทำอย่างไรเมื่อ likelihood ของทุกอนุภาคเป็นศูนย์  
คำใบ้: P_sensor คืน 0 เมื่อ abs(x - y) > 2 ลองดูโค้ดของ monte_carlo_localization ด้วย psource

In [ ]:
# ข้อ 4: รันแล้วสังเกต จากนั้นตอบคำถามในคอมเมนต์
random.seed(11)
try:
    S_bad = monte_carlo_localization({'v': (0, 0), 'w': 0}, (9, 9, 9, 9),
                                     500, P_motion_sample, P_sensor, warehouse)
    print('จำนวนช่องที่มีอนุภาค:', len({(x, y) for x, y, _ in S_bad}))
    for cell, pct in confidence(S_bad, 5):
        print('   %s : %.1f%%' % (cell, pct))
except IndexError as e:
    print('monte_carlo_localization ล้มด้วย IndexError: %r' % (e,))
    print('อ่านโค้ดด้วย psource(monte_carlo_localization) แล้วหาว่าบรรทัดไหนพัง')

# วินิจฉัยก่อนโทษโค้ด: อัลกอริทึมคูณ likelihood ของเซ็นเซอร์ทั้ง 4 ทิศเข้าด้วยกัน
random.seed(11)
particles = [warehouse.sample() for _ in range(500)]
weights = []
for kin in particles:
    w = 1.0
    for j in range(4):
        w *= P_sensor(9, warehouse.ray_cast(j, kin))
    weights.append(w)

print('อนุภาคที่มีน้ำหนักมากกว่าศูนย์: %d จาก %d' % (sum(1 for w in weights if w > 0), len(weights)))
print('ผลรวมน้ำหนักทั้งหมด =', sum(weights))

# TODO ตอบ:
# 1. เกิดอะไรขึ้น: monte_carlo_localization เกิด IndexError บรรทัด W_[i] = W_[i] * P_sensor(z[j], z_) เนื่องจาก z = (9,9,9,9) ซึ่งเป็นค่าที่ไม่ถูกต้อง
# 2. ระบบจริงควร: มี outlier rejection ใส่ค่าfloor เล็ก ๆ และ เข้าโหมด global relocalization ถ้าเกิดติดกันหลายครั้ง

# ข้อ 5: เปลี่ยนตัวเลขต้นทุน เปลี่ยนทั้งระบบ
โรงงานย้ายไปผลิตยาฉีด ซึ่งความเสียหายจากเครื่องพังกลางกะพุ่งเป็น 5,000,000 บาท ส่วนค่าซ่อมยังเท่าเดิม

1.เกณฑ์ break-even ใหม่เป็นเท่าไร  
2.ระบบจะสั่งหยุดซ่อมที่กะไหน  
3.VPI ที่กะ 4 เปลี่ยนไปอย่างไร และยังคุ้มค่าสแกน 8,000 บาทไหม
คำใบ้: เขียนฟังก์ชันรับ cost_breakdown เป็นพารามิเตอร์ แทนการแก้ตัวแปร global

In [ ]:
# ข้อ 5: เติมโค้ดตรง TODO
def make_decision_model(cost_breakdown, cost_repair=COST_REPAIR):
    # คืน (eu, meu, breakeven) สำหรับต้นทุนชุดใหม่
    U = {('repair', True): -cost_repair,   ('repair', False): -cost_repair,
         ('wait', True): -cost_breakdown,  ('wait', False): 0}

    def eu(a, p):
        return p * U[(a, True)] + (1 - p) * U[(a, False)]

    def meu_(p):
        return max((eu(a, p), a) for a in ACTIONS)

    return eu, meu_, cost_repair / cost_breakdown


# TODO: สร้างโมเดลต้นทุนใหม่ที่ 5,000,000 บาท แล้วตอบคำถามทั้ง 3 ข้อ
eu5, meu5, be5 = make_decision_model(5_000_000)
# 1. เกณฑ์ break-even ใหม่ คือ 0.01
# 2. กะที่ 1
# 3. กลายเป็น 0 บาท ไม่คุ้มสแกน

# ข้อ 6 (ท้าทาย): เรียนพารามิเตอร์จากข้อมูล
ที่ผ่านมาเรากรอก maint_transition และ maint_sensor ด้วยมือ ในงานจริงต้องเรียนจากข้อมูล

ถ้ามีข้อมูลที่ รู้สถานะจริง (จากใบบันทึกการซ่อม) การประมาณค่าทำได้ด้วยการนับตรง ๆ เขียนฟังก์ชัน fit_hmm(states, evidence) ที่รับลิสต์สถานะจริงกับลิสต์หลักฐาน แล้วคืน transition model กับ sensor model โดยใช้ Laplace smoothing (บวก 1 ทุกช่อง) เพื่อกันความน่าจะเป็นเป็นศูนย์

ทดสอบโดยสร้างข้อมูลจาก pump ตัวจริง 5,000 กะ แล้วดูว่าค่าที่เรียนได้เข้าใกล้ของจริงไหม

คำใบ้: transition นับคู่ (states[i], states[i+1]), sensor นับคู่ (states[i], evidence[i]) Laplace smoothing คือ (count + 1) / (total + จำนวนค่าที่เป็นไปได้)

In [ ]:
# ข้อ 6: เติมโค้ดตรง TODO
def sample_hmm(hmm, n, seed= 0):
    # สร้างข้อมูลจำลองจาก HMM: คืน (states, evidence) โดย state True = Healthy
    random.seed(seed)
    states, evidence = [], []
    healthy = probability(hmm.prior[0])
    for _ in range(n):
        row = hmm.transition_model[0 if healthy else 1]
        healthy = probability(row[0])
        states.append(healthy)
        p_alert = hmm.sensor_model[0][0 if healthy else 1]
        evidence.append(probability(p_alert))
    return states, evidence


def fit_hmm(states, evidence):
    tc = {(a, b): 1 for a in (True, False) for b in (True, False)}
    sc = {(a, b): 1 for a in (True, False) for b in (True, False)}

    for i in range(len(states) - 1):
        tc[(states[i], states[i + 1])] += 1
        
    for s, e in zip(states, evidence):
        sc[(s, e)] += 1

    def row(counts, key):
        tot = counts[(key, True)] + counts[(key, False)]
        return [counts[(key, True)] / tot, counts[(key, False)] / tot]

    trans = [row(tc, True), row(tc, False)]
    p_alert = [
        sc[(True, True)] / (sc[(True, True)] + sc[(True, False)]),
        sc[(False, True)] / (sc[(False, True)] + sc[(False, False)])
    ]
    
    sensor = [p_alert, [1 - p_alert[0], 1 - p_alert[1]]]

    return trans, sensor


states, evidence = sample_hmm(pump, 5000)
learned = fit_hmm(states, evidence)
print('transition ที่เรียนได้:', learned[0] if learned else None)
print('transition จริง      :', maint_transition)
print('sensor ที่เรียนได้   :', learned[1] if learned else None)
print('sensor จริง          :', maint_sensor)
     